# P05: run OMPar on PolyBench's loops (Phase 2.5 item 4)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

It installs OMPar at a pinned commit in its own Python 3.11 environment, downloads its
two models (about 4 GB: the classifier from the authors' Google Drive folder, the
generator from HuggingFace), and asks it about every `for` loop in the 30 kernels. The
last cell downloads **`results.jsonl`**: give that file to Claude.

OMPar's repository has no licence file, so nothing of it is copied into this project;
the notebook fetches it. Its classifier checkpoint is a pickle, which is one more reason
to run this on a throwaway Colab machine.

Generated by `eval/llm_baseline/ompar/make_notebook.py`: edit that, not this notebook.


In [ ]:
%%bash
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
free -g | head -2; df -h /content | tail -1


In [ ]:
%%bash
set -e
cd /content
[ -d OMPar ] || git clone -q https://github.com/Scientific-Computing-Lab/OMPar
git -C OMPar checkout -q 8eea3f8b7a1966bc5241291b484b13cb80af9e1a
git -C OMPar rev-parse HEAD
pip install -q uv
uv venv --python 3.11 --seed /content/venv
uv pip install -q --python /content/venv/bin/python torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
uv pip install -q --python /content/venv/bin/python \
  transformers==4.44.1 tokenizers==0.19.1 huggingface_hub==0.24.6 safetensors==0.4.4 \
  numpy==1.26.4 tree-sitter==0.20.4 gdown


In [ ]:
%%bash
set -e
cd /content
[ -f weights/model.bin ] || /content/venv/bin/gdown --folder -O weights https://drive.google.com/drive/folders/1tnJf9YvjpDLktVi23TkW-rpjqfdZoybf
ls -l weights; sha256sum weights/model.bin
/content/venv/bin/python - <<'PY'
from huggingface_hub import HfApi
sha = HfApi().model_info('MonoCoder/MonoCoder_OMP').sha
print('MonoCoder_OMP', sha)
assert sha == '938a064aaccf065c795aac7ba76a0f76e6044169', 'the generator changed since this notebook was written'
PY


In [ ]:
%%writefile /content/ompar_run.py
#!/usr/bin/env python3
"""Run OMPar (unmodified, commit pinned by the notebook) over loops.jsonl.

    ompar_run.py --model_weights <dir with model.bin> loops.jsonl results.jsonl

Run from the OMPar checkout. OMPar's gen_par calls generate(max_length=64),
and that limit includes the input: with the pinned transformers (4.44.1) a
loop of more than 64 tokens makes generate() raise ValueError, so OMPar as
published gives no pragma for it however good the model is (recorded as
`pragma: null` with the error in `published_error`). Each loop
therefore gets both answers: `pragma` from OMPar exactly as published, and
`pragma_long` with the same classifier decision and pragma_format but
max_new_tokens=64, decoding only the generated tokens. `tokens` is the
input's length, so the report can say how many loops the limit affected
(186 of the 333 loops have 64 or more).
"""
import argparse
import json

import torch
from compAI import OMPAR


def long_pragma(ompar, loop):
    inputs = ompar.tokenizer_gen(loop, return_tensors="pt").to(ompar.device)
    outputs = ompar.model_gen.generate(inputs["input_ids"], max_new_tokens=64)
    # Decode only the new tokens: gen_par's text[len(loop):] is off by one
    # character when decoding cleans up spaces in the loop (" ?").
    new = outputs[0][inputs["input_ids"].shape[-1]:]
    return ompar.pragma_format(ompar.tokenizer_gen.decode(new, skip_special_tokens=True))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--vocab_file", default="tokenizer/gpt/gpt_vocab/gpt2-vocab.json")
    ap.add_argument("--merge_file", default="tokenizer/gpt/gpt_vocab/gpt2-merges.txt")
    ap.add_argument("--model_weights", required=True)
    ap.add_argument("loops")
    ap.add_argument("results")
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device, flush=True)
    ompar = OMPAR(model_path=args.model_weights, device=device, args=args)
    rows = [json.loads(l) for l in open(args.loops)]
    with open(args.results, "w") as f, torch.no_grad():
        for i, r in enumerate(rows):
            loop = r["code"]
            parallel = bool(ompar.cls_par(loop))
            published, error = None, None
            if parallel:
                try:
                    published = ompar.pragma_format(ompar.gen_par(loop))
                except ValueError as e:  # the input is longer than max_length
                    error = str(e).split(".")[0]
            r2 = {"kernel": r["kernel"], "line": r["line"], "parallel": parallel,
                  "tokens": len(ompar.tokenizer_gen(loop)["input_ids"]),
                  "pragma": published, "published_error": error,
                  "pragma_long": long_pragma(ompar, loop) if parallel else None}
            f.write(json.dumps(r2) + "\n")
            f.flush()
            if i % 20 == 0:
                print(f"{i + 1}/{len(rows)} {r['kernel']}:{r['line']} -> {r2['pragma']}", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/loops.jsonl
{"kernel": "correlation", "line": 35, "code": "for (i = 0; i < N; i++)\n  for (j = 0; j < M; j++)\n    data[i][j] = (DATA_TYPE)(i*j)/M + i;"}
{"kernel": "correlation", "line": 36, "code": "for (j = 0; j < M; j++)\n  data[i][j] = (DATA_TYPE)(i*j)/M + i;"}
{"kernel": "correlation", "line": 53, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < m; j++) {\n    if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, corr[i][j]);\n  }"}
{"kernel": "correlation", "line": 54, "code": "for (j = 0; j < m; j++) {\n  if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, corr[i][j]);\n}"}
{"kernel": "correlation", "line": 79, "code": "for (j = 0; j < _PB_M; j++)\n  {\n    mean[j] = SCALAR_VAL(0.0);\n    for (i = 0; i < _PB_N; i++)\n      mean[j] += data[i][j];\n    mean[j] /= float_n;\n  }"}
{"kernel": "correlation", "line": 82, "code": "for (i = 0; i < _PB_N; i++)\n  mean[j] += data[i][j];"}
{"kernel": "correlation", "line": 88, "code": "for (j = 0; j < _PB_M; j++)\n {\n   stddev[j] = SCALAR_VAL(0.0);\n   for (i = 0; i < _PB_N; i++)\n     stddev[j] += (data[i][j] - mean[j]) * (data[i][j] - mean[j]);\n   stddev[j] /= float_n;\n   stddev[j] = SQRT_FUN(stddev[j]);\n   /* The following in an inelegant but usual way to handle\n      near-zero std. dev. values, which below would cause a zero-\n      divide. */\n   stddev[j] = stddev[j] <= eps ? SCALAR_VAL(1.0) : stddev[j];\n }"}
{"kernel": "correlation", "line": 91, "code": "for (i = 0; i < _PB_N; i++)\n  stddev[j] += (data[i][j] - mean[j]) * (data[i][j] - mean[j]);"}
{"kernel": "correlation", "line": 102, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_M; j++)\n    {\n      data[i][j] -= mean[j];\n      data[i][j] /= SQRT_FUN(float_n) * stddev[j];\n    }"}
{"kernel": "correlation", "line": 103, "code": "for (j = 0; j < _PB_M; j++)\n  {\n    data[i][j] -= mean[j];\n    data[i][j] /= SQRT_FUN(float_n) * stddev[j];\n  }"}
{"kernel": "correlation", "line": 110, "code": "for (i = 0; i < _PB_M-1; i++)\n  {\n    corr[i][i] = SCALAR_VAL(1.0);\n    for (j = i+1; j < _PB_M; j++)\n      {\n        corr[i][j] = SCALAR_VAL(0.0);\n        for (k = 0; k < _PB_N; k++)\n          corr[i][j] += (data[k][i] * data[k][j]);\n        corr[j][i] = corr[i][j];\n      }\n  }"}
{"kernel": "correlation", "line": 113, "code": "for (j = i+1; j < _PB_M; j++)\n  {\n    corr[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_N; k++)\n      corr[i][j] += (data[k][i] * data[k][j]);\n    corr[j][i] = corr[i][j];\n  }"}
{"kernel": "correlation", "line": 116, "code": "for (k = 0; k < _PB_N; k++)\n  corr[i][j] += (data[k][i] * data[k][j]);"}
{"kernel": "covariance", "line": 34, "code": "for (i = 0; i < N; i++)\n  for (j = 0; j < M; j++)\n    data[i][j] = ((DATA_TYPE) i*j) / M;"}
{"kernel": "covariance", "line": 35, "code": "for (j = 0; j < M; j++)\n  data[i][j] = ((DATA_TYPE) i*j) / M;"}
{"kernel": "covariance", "line": 51, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < m; j++) {\n    if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, cov[i][j]);\n  }"}
{"kernel": "covariance", "line": 52, "code": "for (j = 0; j < m; j++) {\n  if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, cov[i][j]);\n}"}
{"kernel": "covariance", "line": 73, "code": "for (j = 0; j < _PB_M; j++)\n  {\n    mean[j] = SCALAR_VAL(0.0);\n    for (i = 0; i < _PB_N; i++)\n      mean[j] += data[i][j];\n    mean[j] /= float_n;\n  }"}
{"kernel": "covariance", "line": 76, "code": "for (i = 0; i < _PB_N; i++)\n  mean[j] += data[i][j];"}
{"kernel": "covariance", "line": 81, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_M; j++)\n    data[i][j] -= mean[j];"}
{"kernel": "covariance", "line": 82, "code": "for (j = 0; j < _PB_M; j++)\n  data[i][j] -= mean[j];"}
{"kernel": "covariance", "line": 85, "code": "for (i = 0; i < _PB_M; i++)\n  for (j = i; j < _PB_M; j++)\n    {\n      cov[i][j] = SCALAR_VAL(0.0);\n      for (k = 0; k < _PB_N; k++)\n        cov[i][j] += data[k][i] * data[k][j];\n      cov[i][j] /= (float_n - SCALAR_VAL(1.0));\n      cov[j][i] = cov[i][j];\n    }"}
{"kernel": "covariance", "line": 86, "code": "for (j = i; j < _PB_M; j++)\n  {\n    cov[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_N; k++)\n      cov[i][j] += data[k][i] * data[k][j];\n    cov[i][j] /= (float_n - SCALAR_VAL(1.0));\n    cov[j][i] = cov[i][j];\n  }"}
{"kernel": "covariance", "line": 89, "code": "for (k = 0; k < _PB_N; k++)\n  cov[i][j] += data[k][i] * data[k][j];"}
{"kernel": "gemm", "line": 37, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nj; j++)\n    C[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;"}
{"kernel": "gemm", "line": 38, "code": "for (j = 0; j < nj; j++)\n  C[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;"}
{"kernel": "gemm", "line": 40, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nk; j++)\n    A[i][j] = (DATA_TYPE) (i*(j+1) % nk) / nk;"}
{"kernel": "gemm", "line": 41, "code": "for (j = 0; j < nk; j++)\n  A[i][j] = (DATA_TYPE) (i*(j+1) % nk) / nk;"}
{"kernel": "gemm", "line": 43, "code": "for (i = 0; i < nk; i++)\n  for (j = 0; j < nj; j++)\n    B[i][j] = (DATA_TYPE) (i*(j+2) % nj) / nj;"}
{"kernel": "gemm", "line": 44, "code": "for (j = 0; j < nj; j++)\n  B[i][j] = (DATA_TYPE) (i*(j+2) % nj) / nj;"}
{"kernel": "gemm", "line": 59, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nj; j++) {\n      if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n  }"}
{"kernel": "gemm", "line": 60, "code": "for (j = 0; j < nj; j++) {\n    if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n}"}
{"kernel": "gemm", "line": 89, "code": "for (i = 0; i < _PB_NI; i++) {\n  for (j = 0; j < _PB_NJ; j++)\n      C[i][j] *= beta;\n  for (k = 0; k < _PB_NK; k++) {\n     for (j = 0; j < _PB_NJ; j++)\n        C[i][j] += alpha * A[i][k] * B[k][j];\n  }\n}"}
{"kernel": "gemm", "line": 90, "code": "for (j = 0; j < _PB_NJ; j++)\n    C[i][j] *= beta;"}
{"kernel": "gemm", "line": 92, "code": "for (k = 0; k < _PB_NK; k++) {\n   for (j = 0; j < _PB_NJ; j++)\n      C[i][j] += alpha * A[i][k] * B[k][j];\n}"}
{"kernel": "gemm", "line": 93, "code": "for (j = 0; j < _PB_NJ; j++)\n   C[i][j] += alpha * A[i][k] * B[k][j];"}
{"kernel": "gemver", "line": 46, "code": "for (i = 0; i < n; i++)\n  {\n    u1[i] = i;\n    u2[i] = ((i+1)/fn)/2.0;\n    v1[i] = ((i+1)/fn)/4.0;\n    v2[i] = ((i+1)/fn)/6.0;\n    y[i] = ((i+1)/fn)/8.0;\n    z[i] = ((i+1)/fn)/9.0;\n    x[i] = 0.0;\n    w[i] = 0.0;\n    for (j = 0; j < n; j++)\n      A[i][j] = (DATA_TYPE) (i*j % n) / n;\n  }"}
{"kernel": "gemver", "line": 56, "code": "for (j = 0; j < n; j++)\n  A[i][j] = (DATA_TYPE) (i*j % n) / n;"}
{"kernel": "gemver", "line": 72, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, w[i]);\n}"}
{"kernel": "gemver", "line": 101, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    A[i][j] = A[i][j] + u1[i] * v1[j] + u2[i] * v2[j];"}
{"kernel": "gemver", "line": 102, "code": "for (j = 0; j < _PB_N; j++)\n  A[i][j] = A[i][j] + u1[i] * v1[j] + u2[i] * v2[j];"}
{"kernel": "gemver", "line": 105, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    x[i] = x[i] + beta * A[j][i] * y[j];"}
{"kernel": "gemver", "line": 106, "code": "for (j = 0; j < _PB_N; j++)\n  x[i] = x[i] + beta * A[j][i] * y[j];"}
{"kernel": "gemver", "line": 109, "code": "for (i = 0; i < _PB_N; i++)\n  x[i] = x[i] + z[i];"}
{"kernel": "gemver", "line": 112, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    w[i] = w[i] +  alpha * A[i][j] * x[j];"}
{"kernel": "gemver", "line": 113, "code": "for (j = 0; j < _PB_N; j++)\n  w[i] = w[i] +  alpha * A[i][j] * x[j];"}
{"kernel": "gesummv", "line": 37, "code": "for (i = 0; i < n; i++)\n  {\n    x[i] = (DATA_TYPE)( i % n) / n;\n    for (j = 0; j < n; j++) {\n      A[i][j] = (DATA_TYPE) ((i*j+1) % n) / n;\n      B[i][j] = (DATA_TYPE) ((i*j+2) % n) / n;\n    }\n  }"}
{"kernel": "gesummv", "line": 40, "code": "for (j = 0; j < n; j++) {\n  A[i][j] = (DATA_TYPE) ((i*j+1) % n) / n;\n  B[i][j] = (DATA_TYPE) ((i*j+2) % n) / n;\n}"}
{"kernel": "gesummv", "line": 59, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\n}"}
{"kernel": "gesummv", "line": 83, "code": "for (i = 0; i < _PB_N; i++)\n  {\n    tmp[i] = SCALAR_VAL(0.0);\n    y[i] = SCALAR_VAL(0.0);\n    for (j = 0; j < _PB_N; j++)\n      {\n        tmp[i] = A[i][j] * x[j] + tmp[i];\n        y[i] = B[i][j] * x[j] + y[i];\n      }\n    y[i] = alpha * tmp[i] + beta * y[i];\n  }"}
{"kernel": "gesummv", "line": 87, "code": "for (j = 0; j < _PB_N; j++)\n  {\n    tmp[i] = A[i][j] * x[j] + tmp[i];\n    y[i] = B[i][j] * x[j] + y[i];\n  }"}
{"kernel": "symm", "line": 37, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++) {\n    C[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\n    B[i][j] = (DATA_TYPE) ((n+i-j) % 100) / m;\n  }"}
{"kernel": "symm", "line": 38, "code": "for (j = 0; j < n; j++) {\n  C[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\n  B[i][j] = (DATA_TYPE) ((n+i-j) % 100) / m;\n}"}
{"kernel": "symm", "line": 42, "code": "for (i = 0; i < m; i++) {\n  for (j = 0; j <=i; j++)\n    A[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\n  for (j = i+1; j < m; j++)\n    A[i][j] = -999; //regions of arrays that should not be used\n}"}
{"kernel": "symm", "line": 43, "code": "for (j = 0; j <=i; j++)\n  A[i][j] = (DATA_TYPE) ((i+j) % 100) / m;"}
{"kernel": "symm", "line": 45, "code": "for (j = i+1; j < m; j++)\n  A[i][j] = -999; //regions of arrays that should not be used"}
{"kernel": "symm", "line": 61, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++) {\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n  }"}
{"kernel": "symm", "line": 62, "code": "for (j = 0; j < n; j++) {\n    if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n}"}
{"kernel": "symm", "line": 93, "code": "for (i = 0; i < _PB_M; i++)\n   for (j = 0; j < _PB_N; j++ )\n   {\n     temp2 = 0;\n     for (k = 0; k < i; k++) {\n        C[k][j] += alpha*B[i][j] * A[i][k];\n        temp2 += B[k][j] * A[i][k];\n     }\n     C[i][j] = beta * C[i][j] + alpha*B[i][j] * A[i][i] + alpha * temp2;\n  }"}
{"kernel": "symm", "line": 94, "code": " for (j = 0; j < _PB_N; j++ )\n {\n   temp2 = 0;\n   for (k = 0; k < i; k++) {\n      C[k][j] += alpha*B[i][j] * A[i][k];\n      temp2 += B[k][j] * A[i][k];\n   }\n   C[i][j] = beta * C[i][j] + alpha*B[i][j] * A[i][i] + alpha * temp2;\n}"}
{"kernel": "symm", "line": 97, "code": "for (k = 0; k < i; k++) {\n   C[k][j] += alpha*B[i][j] * A[i][k];\n   temp2 += B[k][j] * A[i][k];\n}"}
{"kernel": "syr2k", "line": 37, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < m; j++) {\n    A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\n    B[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\n  }"}
{"kernel": "syr2k", "line": 38, "code": "for (j = 0; j < m; j++) {\n  A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\n  B[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\n}"}
{"kernel": "syr2k", "line": 42, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    C[i][j] = (DATA_TYPE) ((i*j+3)%n) / m;\n  }"}
{"kernel": "syr2k", "line": 43, "code": "for (j = 0; j < n; j++) {\n  C[i][j] = (DATA_TYPE) ((i*j+3)%n) / m;\n}"}
{"kernel": "syr2k", "line": 59, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n  }"}
{"kernel": "syr2k", "line": 60, "code": "for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n}"}
{"kernel": "syr2k", "line": 88, "code": "for (i = 0; i < _PB_N; i++) {\n  for (j = 0; j <= i; j++)\n    C[i][j] *= beta;\n  for (k = 0; k < _PB_M; k++)\n    for (j = 0; j <= i; j++)\n      {\n        C[i][j] += A[j][k]*alpha*B[i][k] + B[j][k]*alpha*A[i][k];\n      }\n}"}
{"kernel": "syr2k", "line": 89, "code": "for (j = 0; j <= i; j++)\n  C[i][j] *= beta;"}
{"kernel": "syr2k", "line": 91, "code": "for (k = 0; k < _PB_M; k++)\n  for (j = 0; j <= i; j++)\n    {\n      C[i][j] += A[j][k]*alpha*B[i][k] + B[j][k]*alpha*A[i][k];\n    }"}
{"kernel": "syr2k", "line": 92, "code": "for (j = 0; j <= i; j++)\n  {\n    C[i][j] += A[j][k]*alpha*B[i][k] + B[j][k]*alpha*A[i][k];\n  }"}
{"kernel": "syrk", "line": 36, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < m; j++)\n    A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;"}
{"kernel": "syrk", "line": 37, "code": "for (j = 0; j < m; j++)\n  A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;"}
{"kernel": "syrk", "line": 39, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    C[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;"}
{"kernel": "syrk", "line": 40, "code": "for (j = 0; j < n; j++)\n  C[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;"}
{"kernel": "syrk", "line": 55, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n  }"}
{"kernel": "syrk", "line": 56, "code": "for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\n}"}
{"kernel": "syrk", "line": 83, "code": "for (i = 0; i < _PB_N; i++) {\n  for (j = 0; j <= i; j++)\n    C[i][j] *= beta;\n  for (k = 0; k < _PB_M; k++) {\n    for (j = 0; j <= i; j++)\n      C[i][j] += alpha * A[i][k] * A[j][k];\n  }\n}"}
{"kernel": "syrk", "line": 84, "code": "for (j = 0; j <= i; j++)\n  C[i][j] *= beta;"}
{"kernel": "syrk", "line": 86, "code": "for (k = 0; k < _PB_M; k++) {\n  for (j = 0; j <= i; j++)\n    C[i][j] += alpha * A[i][k] * A[j][k];\n}"}
{"kernel": "syrk", "line": 87, "code": "for (j = 0; j <= i; j++)\n  C[i][j] += alpha * A[i][k] * A[j][k];"}
{"kernel": "trmm", "line": 34, "code": " for (i = 0; i < m; i++) {\n   for (j = 0; j < i; j++) {\n     A[i][j] = (DATA_TYPE)((i+j) % m)/m;\n   }\n   A[i][i] = 1.0;\n   for (j = 0; j < n; j++) {\n     B[i][j] = (DATA_TYPE)((n+(i-j)) % n)/n;\n   }\n}"}
{"kernel": "trmm", "line": 35, "code": "for (j = 0; j < i; j++) {\n  A[i][j] = (DATA_TYPE)((i+j) % m)/m;\n}"}
{"kernel": "trmm", "line": 39, "code": "for (j = 0; j < n; j++) {\n  B[i][j] = (DATA_TYPE)((n+(i-j)) % n)/n;\n}"}
{"kernel": "trmm", "line": 57, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++) {\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, B[i][j]);\n  }"}
{"kernel": "trmm", "line": 58, "code": "for (j = 0; j < n; j++) {\n    if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, B[i][j]);\n}"}
{"kernel": "trmm", "line": 86, "code": "for (i = 0; i < _PB_M; i++)\n   for (j = 0; j < _PB_N; j++) {\n      for (k = i+1; k < _PB_M; k++)\n         B[i][j] += A[k][i] * B[k][j];\n      B[i][j] = alpha * B[i][j];\n   }"}
{"kernel": "trmm", "line": 87, "code": "for (j = 0; j < _PB_N; j++) {\n   for (k = i+1; k < _PB_M; k++)\n      B[i][j] += A[k][i] * B[k][j];\n   B[i][j] = alpha * B[i][j];\n}"}
{"kernel": "trmm", "line": 88, "code": "for (k = i+1; k < _PB_M; k++)\n   B[i][j] += A[k][i] * B[k][j];"}
{"kernel": "2mm", "line": 38, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nk; j++)\n    A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;"}
{"kernel": "2mm", "line": 39, "code": "for (j = 0; j < nk; j++)\n  A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;"}
{"kernel": "2mm", "line": 41, "code": "for (i = 0; i < nk; i++)\n  for (j = 0; j < nj; j++)\n    B[i][j] = (DATA_TYPE) (i*(j+1) % nj) / nj;"}
{"kernel": "2mm", "line": 42, "code": "for (j = 0; j < nj; j++)\n  B[i][j] = (DATA_TYPE) (i*(j+1) % nj) / nj;"}
{"kernel": "2mm", "line": 44, "code": "for (i = 0; i < nj; i++)\n  for (j = 0; j < nl; j++)\n    C[i][j] = (DATA_TYPE) ((i*(j+3)+1) % nl) / nl;"}
{"kernel": "2mm", "line": 45, "code": "for (j = 0; j < nl; j++)\n  C[i][j] = (DATA_TYPE) ((i*(j+3)+1) % nl) / nl;"}
{"kernel": "2mm", "line": 47, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nl; j++)\n    D[i][j] = (DATA_TYPE) (i*(j+2) % nk) / nk;"}
{"kernel": "2mm", "line": 48, "code": "for (j = 0; j < nl; j++)\n  D[i][j] = (DATA_TYPE) (i*(j+2) % nk) / nk;"}
{"kernel": "2mm", "line": 63, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nl; j++) {\n      if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, D[i][j]);\n  }"}
{"kernel": "2mm", "line": 64, "code": "for (j = 0; j < nl; j++) {\n    if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, D[i][j]);\n}"}
{"kernel": "2mm", "line": 89, "code": "for (i = 0; i < _PB_NI; i++)\n  for (j = 0; j < _PB_NJ; j++)\n    {\n      tmp[i][j] = SCALAR_VAL(0.0);\n      for (k = 0; k < _PB_NK; ++k)\n        tmp[i][j] += alpha * A[i][k] * B[k][j];\n    }"}
{"kernel": "2mm", "line": 90, "code": "for (j = 0; j < _PB_NJ; j++)\n  {\n    tmp[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_NK; ++k)\n      tmp[i][j] += alpha * A[i][k] * B[k][j];\n  }"}
{"kernel": "2mm", "line": 93, "code": "for (k = 0; k < _PB_NK; ++k)\n  tmp[i][j] += alpha * A[i][k] * B[k][j];"}
{"kernel": "2mm", "line": 96, "code": "for (i = 0; i < _PB_NI; i++)\n  for (j = 0; j < _PB_NL; j++)\n    {\n      D[i][j] *= beta;\n      for (k = 0; k < _PB_NJ; ++k)\n        D[i][j] += tmp[i][k] * C[k][j];\n    }"}
{"kernel": "2mm", "line": 97, "code": "for (j = 0; j < _PB_NL; j++)\n  {\n    D[i][j] *= beta;\n    for (k = 0; k < _PB_NJ; ++k)\n      D[i][j] += tmp[i][k] * C[k][j];\n  }"}
{"kernel": "2mm", "line": 100, "code": "for (k = 0; k < _PB_NJ; ++k)\n  D[i][j] += tmp[i][k] * C[k][j];"}
{"kernel": "3mm", "line": 34, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nk; j++)\n    A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / (5*ni);"}
{"kernel": "3mm", "line": 35, "code": "for (j = 0; j < nk; j++)\n  A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / (5*ni);"}
{"kernel": "3mm", "line": 37, "code": "for (i = 0; i < nk; i++)\n  for (j = 0; j < nj; j++)\n    B[i][j] = (DATA_TYPE) ((i*(j+1)+2) % nj) / (5*nj);"}
{"kernel": "3mm", "line": 38, "code": "for (j = 0; j < nj; j++)\n  B[i][j] = (DATA_TYPE) ((i*(j+1)+2) % nj) / (5*nj);"}
{"kernel": "3mm", "line": 40, "code": "for (i = 0; i < nj; i++)\n  for (j = 0; j < nm; j++)\n    C[i][j] = (DATA_TYPE) (i*(j+3) % nl) / (5*nl);"}
{"kernel": "3mm", "line": 41, "code": "for (j = 0; j < nm; j++)\n  C[i][j] = (DATA_TYPE) (i*(j+3) % nl) / (5*nl);"}
{"kernel": "3mm", "line": 43, "code": "for (i = 0; i < nm; i++)\n  for (j = 0; j < nl; j++)\n    D[i][j] = (DATA_TYPE) ((i*(j+2)+2) % nk) / (5*nk);"}
{"kernel": "3mm", "line": 44, "code": "for (j = 0; j < nl; j++)\n  D[i][j] = (DATA_TYPE) ((i*(j+2)+2) % nk) / (5*nk);"}
{"kernel": "3mm", "line": 59, "code": "for (i = 0; i < ni; i++)\n  for (j = 0; j < nl; j++) {\n      if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, G[i][j]);\n  }"}
{"kernel": "3mm", "line": 60, "code": "for (j = 0; j < nl; j++) {\n    if ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, G[i][j]);\n}"}
{"kernel": "3mm", "line": 85, "code": "for (i = 0; i < _PB_NI; i++)\n  for (j = 0; j < _PB_NJ; j++)\n    {\n      E[i][j] = SCALAR_VAL(0.0);\n      for (k = 0; k < _PB_NK; ++k)\n        E[i][j] += A[i][k] * B[k][j];\n    }"}
{"kernel": "3mm", "line": 86, "code": "for (j = 0; j < _PB_NJ; j++)\n  {\n    E[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_NK; ++k)\n      E[i][j] += A[i][k] * B[k][j];\n  }"}
{"kernel": "3mm", "line": 89, "code": "for (k = 0; k < _PB_NK; ++k)\n  E[i][j] += A[i][k] * B[k][j];"}
{"kernel": "3mm", "line": 93, "code": "for (i = 0; i < _PB_NJ; i++)\n  for (j = 0; j < _PB_NL; j++)\n    {\n      F[i][j] = SCALAR_VAL(0.0);\n      for (k = 0; k < _PB_NM; ++k)\n        F[i][j] += C[i][k] * D[k][j];\n    }"}
{"kernel": "3mm", "line": 94, "code": "for (j = 0; j < _PB_NL; j++)\n  {\n    F[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_NM; ++k)\n      F[i][j] += C[i][k] * D[k][j];\n  }"}
{"kernel": "3mm", "line": 97, "code": "for (k = 0; k < _PB_NM; ++k)\n  F[i][j] += C[i][k] * D[k][j];"}
{"kernel": "3mm", "line": 101, "code": "for (i = 0; i < _PB_NI; i++)\n  for (j = 0; j < _PB_NL; j++)\n    {\n      G[i][j] = SCALAR_VAL(0.0);\n      for (k = 0; k < _PB_NJ; ++k)\n        G[i][j] += E[i][k] * F[k][j];\n    }"}
{"kernel": "3mm", "line": 102, "code": "for (j = 0; j < _PB_NL; j++)\n  {\n    G[i][j] = SCALAR_VAL(0.0);\n    for (k = 0; k < _PB_NJ; ++k)\n      G[i][j] += E[i][k] * F[k][j];\n  }"}
{"kernel": "3mm", "line": 105, "code": "for (k = 0; k < _PB_NJ; ++k)\n  G[i][j] += E[i][k] * F[k][j];"}
{"kernel": "atax", "line": 34, "code": "for (i = 0; i < n; i++)\n    x[i] = 1 + (i / fn);"}
{"kernel": "atax", "line": 36, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++)\n    A[i][j] = (DATA_TYPE) ((i+j) % n) / (5*m);"}
{"kernel": "atax", "line": 37, "code": "for (j = 0; j < n; j++)\n  A[i][j] = (DATA_TYPE) ((i+j) % n) / (5*m);"}
{"kernel": "atax", "line": 53, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\n}"}
{"kernel": "atax", "line": 74, "code": "for (i = 0; i < _PB_N; i++)\n  y[i] = 0;"}
{"kernel": "atax", "line": 76, "code": "for (i = 0; i < _PB_M; i++)\n  {\n    tmp[i] = SCALAR_VAL(0.0);\n    for (j = 0; j < _PB_N; j++)\n      tmp[i] = tmp[i] + A[i][j] * x[j];\n    for (j = 0; j < _PB_N; j++)\n      y[j] = y[j] + A[i][j] * tmp[i];\n  }"}
{"kernel": "atax", "line": 79, "code": "for (j = 0; j < _PB_N; j++)\n  tmp[i] = tmp[i] + A[i][j] * x[j];"}
{"kernel": "atax", "line": 81, "code": "for (j = 0; j < _PB_N; j++)\n  y[j] = y[j] + A[i][j] * tmp[i];"}
{"kernel": "bicg", "line": 33, "code": "for (i = 0; i < m; i++)\n  p[i] = (DATA_TYPE)(i % m) / m;"}
{"kernel": "bicg", "line": 35, "code": "for (i = 0; i < n; i++) {\n  r[i] = (DATA_TYPE)(i % n) / n;\n  for (j = 0; j < m; j++)\n    A[i][j] = (DATA_TYPE) (i*(j+1) % n)/n;\n}"}
{"kernel": "bicg", "line": 37, "code": "for (j = 0; j < m; j++)\n  A[i][j] = (DATA_TYPE) (i*(j+1) % n)/n;"}
{"kernel": "bicg", "line": 55, "code": "for (i = 0; i < m; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, s[i]);\n}"}
{"kernel": "bicg", "line": 61, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, q[i]);\n}"}
{"kernel": "bicg", "line": 83, "code": "for (i = 0; i < _PB_M; i++)\n  s[i] = 0;"}
{"kernel": "bicg", "line": 85, "code": "for (i = 0; i < _PB_N; i++)\n  {\n    q[i] = SCALAR_VAL(0.0);\n    for (j = 0; j < _PB_M; j++)\n      {\n        s[j] = s[j] + r[i] * A[i][j];\n        q[i] = q[i] + A[i][j] * p[j];\n      }\n  }"}
{"kernel": "bicg", "line": 88, "code": "for (j = 0; j < _PB_M; j++)\n  {\n    s[j] = s[j] + r[i] * A[i][j];\n    q[i] = q[i] + A[i][j] * p[j];\n  }"}
{"kernel": "doitgen", "line": 32, "code": "for (i = 0; i < nr; i++)\n  for (j = 0; j < nq; j++)\n    for (k = 0; k < np; k++)\n      A[i][j][k] = (DATA_TYPE) ((i*j + k)%np) / np;"}
{"kernel": "doitgen", "line": 33, "code": "for (j = 0; j < nq; j++)\n  for (k = 0; k < np; k++)\n    A[i][j][k] = (DATA_TYPE) ((i*j + k)%np) / np;"}
{"kernel": "doitgen", "line": 34, "code": "for (k = 0; k < np; k++)\n  A[i][j][k] = (DATA_TYPE) ((i*j + k)%np) / np;"}
{"kernel": "doitgen", "line": 36, "code": "for (i = 0; i < np; i++)\n  for (j = 0; j < np; j++)\n    C4[i][j] = (DATA_TYPE) (i*j % np) / np;"}
{"kernel": "doitgen", "line": 37, "code": "for (j = 0; j < np; j++)\n  C4[i][j] = (DATA_TYPE) (i*j % np) / np;"}
{"kernel": "doitgen", "line": 52, "code": "for (i = 0; i < nr; i++)\n  for (j = 0; j < nq; j++)\n    for (k = 0; k < np; k++) {\n      if ((i*nq*np+j*np+k) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n    }"}
{"kernel": "doitgen", "line": 53, "code": "for (j = 0; j < nq; j++)\n  for (k = 0; k < np; k++) {\n    if ((i*nq*np+j*np+k) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n  }"}
{"kernel": "doitgen", "line": 54, "code": "for (k = 0; k < np; k++) {\n  if ((i*nq*np+j*np+k) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n}"}
{"kernel": "doitgen", "line": 73, "code": "for (r = 0; r < _PB_NR; r++)\n  for (q = 0; q < _PB_NQ; q++)  {\n    for (p = 0; p < _PB_NP; p++)  {\n      sum[p] = SCALAR_VAL(0.0);\n      for (s = 0; s < _PB_NP; s++)\n        sum[p] += A[r][q][s] * C4[s][p];\n    }\n    for (p = 0; p < _PB_NP; p++)\n      A[r][q][p] = sum[p];\n  }"}
{"kernel": "doitgen", "line": 74, "code": "for (q = 0; q < _PB_NQ; q++)  {\n  for (p = 0; p < _PB_NP; p++)  {\n    sum[p] = SCALAR_VAL(0.0);\n    for (s = 0; s < _PB_NP; s++)\n      sum[p] += A[r][q][s] * C4[s][p];\n  }\n  for (p = 0; p < _PB_NP; p++)\n    A[r][q][p] = sum[p];\n}"}
{"kernel": "doitgen", "line": 75, "code": "for (p = 0; p < _PB_NP; p++)  {\n  sum[p] = SCALAR_VAL(0.0);\n  for (s = 0; s < _PB_NP; s++)\n    sum[p] += A[r][q][s] * C4[s][p];\n}"}
{"kernel": "doitgen", "line": 77, "code": "for (s = 0; s < _PB_NP; s++)\n  sum[p] += A[r][q][s] * C4[s][p];"}
{"kernel": "doitgen", "line": 80, "code": "for (p = 0; p < _PB_NP; p++)\n  A[r][q][p] = sum[p];"}
{"kernel": "mvt", "line": 35, "code": "for (i = 0; i < n; i++)\n  {\n    x1[i] = (DATA_TYPE) (i % n) / n;\n    x2[i] = (DATA_TYPE) ((i + 1) % n) / n;\n    y_1[i] = (DATA_TYPE) ((i + 3) % n) / n;\n    y_2[i] = (DATA_TYPE) ((i + 4) % n) / n;\n    for (j = 0; j < n; j++)\n      A[i][j] = (DATA_TYPE) (i*j % n) / n;\n  }"}
{"kernel": "mvt", "line": 41, "code": "for (j = 0; j < n; j++)\n  A[i][j] = (DATA_TYPE) (i*j % n) / n;"}
{"kernel": "mvt", "line": 59, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x1[i]);\n}"}
{"kernel": "mvt", "line": 66, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x2[i]);\n}"}
{"kernel": "mvt", "line": 88, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    x1[i] = x1[i] + A[i][j] * y_1[j];"}
{"kernel": "mvt", "line": 89, "code": "for (j = 0; j < _PB_N; j++)\n  x1[i] = x1[i] + A[i][j] * y_1[j];"}
{"kernel": "mvt", "line": 91, "code": "for (i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    x2[i] = x2[i] + A[j][i] * y_2[j];"}
{"kernel": "mvt", "line": 92, "code": "for (j = 0; j < _PB_N; j++)\n  x2[i] = x2[i] + A[j][i] * y_2[j];"}
{"kernel": "cholesky", "line": 31, "code": "for (i = 0; i < n; i++)\n  {\n    for (j = 0; j <= i; j++)\n      A[i][j] = (DATA_TYPE)(-j % n) / n + 1;\n    for (j = i+1; j < n; j++) {\n      A[i][j] = 0;\n    }\n    A[i][i] = 1;\n  }"}
{"kernel": "cholesky", "line": 33, "code": "for (j = 0; j <= i; j++)\n  A[i][j] = (DATA_TYPE)(-j % n) / n + 1;"}
{"kernel": "cholesky", "line": 35, "code": "for (j = i+1; j < n; j++) {\n  A[i][j] = 0;\n}"}
{"kernel": "cholesky", "line": 44, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "cholesky", "line": 45, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "cholesky", "line": 47, "code": "for (t = 0; t < n; ++t)\n  for (r = 0; r < n; ++r)\n    for (s = 0; s < n; ++s)\n      (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "cholesky", "line": 48, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "cholesky", "line": 49, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "cholesky", "line": 51, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "cholesky", "line": 52, "code": "for (s = 0; s < n; ++s)\n  A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "cholesky", "line": 70, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j <= i; j++) {\n  if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n}"}
{"kernel": "cholesky", "line": 71, "code": "  for (j = 0; j <= i; j++) {\n  if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n}"}
{"kernel": "cholesky", "line": 90, "code": "for (i = 0; i < _PB_N; i++) {\n   //j<i\n   for (j = 0; j < i; j++) {\n      for (k = 0; k < j; k++) {\n         A[i][j] -= A[i][k] * A[j][k];\n      }\n      A[i][j] /= A[j][j];\n   }\n   // i==j case\n   for (k = 0; k < i; k++) {\n      A[i][i] -= A[i][k] * A[i][k];\n   }\n   A[i][i] = SQRT_FUN(A[i][i]);\n}"}
{"kernel": "cholesky", "line": 92, "code": "for (j = 0; j < i; j++) {\n   for (k = 0; k < j; k++) {\n      A[i][j] -= A[i][k] * A[j][k];\n   }\n   A[i][j] /= A[j][j];\n}"}
{"kernel": "cholesky", "line": 93, "code": "for (k = 0; k < j; k++) {\n   A[i][j] -= A[i][k] * A[j][k];\n}"}
{"kernel": "cholesky", "line": 99, "code": "for (k = 0; k < i; k++) {\n   A[i][i] -= A[i][k] * A[i][k];\n}"}
{"kernel": "durbin", "line": 31, "code": "for (i = 0; i < n; i++)\n  {\n    r[i] = (n+1-i);\n  }"}
{"kernel": "durbin", "line": 49, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\n}"}
{"kernel": "durbin", "line": 77, "code": "for (k = 1; k < _PB_N; k++) {\n  beta = (1-alpha*alpha)*beta;\n  sum = SCALAR_VAL(0.0);\n  for (i=0; i<k; i++) {\n     sum += r[k-i-1]*y[i];\n  }\n  alpha = - (r[k] + sum)/beta;\n\n  for (i=0; i<k; i++) {\n     z[i] = y[i] + alpha*y[k-i-1];\n  }\n  for (i=0; i<k; i++) {\n    y[i] = z[i];\n  }\n  y[k] = alpha;\n}"}
{"kernel": "durbin", "line": 80, "code": "for (i=0; i<k; i++) {\n   sum += r[k-i-1]*y[i];\n}"}
{"kernel": "durbin", "line": 85, "code": "for (i=0; i<k; i++) {\n   z[i] = y[i] + alpha*y[k-i-1];\n}"}
{"kernel": "durbin", "line": 88, "code": "for (i=0; i<k; i++) {\n  y[i] = z[i];\n}"}
{"kernel": "gramschmidt", "line": 33, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++) {\n    A[i][j] = (((DATA_TYPE) ((i*j) % m) / m )*100) + 10;\n    Q[i][j] = 0.0;\n  }"}
{"kernel": "gramschmidt", "line": 34, "code": "for (j = 0; j < n; j++) {\n  A[i][j] = (((DATA_TYPE) ((i*j) % m) / m )*100) + 10;\n  Q[i][j] = 0.0;\n}"}
{"kernel": "gramschmidt", "line": 38, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    R[i][j] = 0.0;"}
{"kernel": "gramschmidt", "line": 39, "code": "for (j = 0; j < n; j++)\n  R[i][j] = 0.0;"}
{"kernel": "gramschmidt", "line": 56, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n      if ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, R[i][j]);\n  }"}
{"kernel": "gramschmidt", "line": 57, "code": "for (j = 0; j < n; j++) {\n    if ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, R[i][j]);\n}"}
{"kernel": "gramschmidt", "line": 64, "code": "for (i = 0; i < m; i++)\n  for (j = 0; j < n; j++) {\n      if ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, Q[i][j]);\n  }"}
{"kernel": "gramschmidt", "line": 65, "code": "for (j = 0; j < n; j++) {\n    if ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, Q[i][j]);\n}"}
{"kernel": "gramschmidt", "line": 89, "code": "for (k = 0; k < _PB_N; k++)\n  {\n    nrm = SCALAR_VAL(0.0);\n    for (i = 0; i < _PB_M; i++)\n      nrm += A[i][k] * A[i][k];\n    R[k][k] = SQRT_FUN(nrm);\n    for (i = 0; i < _PB_M; i++)\n      Q[i][k] = A[i][k] / R[k][k];\n    for (j = k + 1; j < _PB_N; j++)\n      {\n        R[k][j] = SCALAR_VAL(0.0);\n        for (i = 0; i < _PB_M; i++)\n          R[k][j] += Q[i][k] * A[i][j];\n        for (i = 0; i < _PB_M; i++)\n          A[i][j] = A[i][j] - Q[i][k] * R[k][j];\n      }\n  }"}
{"kernel": "gramschmidt", "line": 92, "code": "for (i = 0; i < _PB_M; i++)\n  nrm += A[i][k] * A[i][k];"}
{"kernel": "gramschmidt", "line": 95, "code": "for (i = 0; i < _PB_M; i++)\n  Q[i][k] = A[i][k] / R[k][k];"}
{"kernel": "gramschmidt", "line": 97, "code": "for (j = k + 1; j < _PB_N; j++)\n  {\n    R[k][j] = SCALAR_VAL(0.0);\n    for (i = 0; i < _PB_M; i++)\n      R[k][j] += Q[i][k] * A[i][j];\n    for (i = 0; i < _PB_M; i++)\n      A[i][j] = A[i][j] - Q[i][k] * R[k][j];\n  }"}
{"kernel": "gramschmidt", "line": 100, "code": "for (i = 0; i < _PB_M; i++)\n  R[k][j] += Q[i][k] * A[i][j];"}
{"kernel": "gramschmidt", "line": 102, "code": "for (i = 0; i < _PB_M; i++)\n  A[i][j] = A[i][j] - Q[i][k] * R[k][j];"}
{"kernel": "lu", "line": 31, "code": "for (i = 0; i < n; i++)\n  {\n    for (j = 0; j <= i; j++)\n      A[i][j] = (DATA_TYPE)(-j % n) / n + 1;\n    for (j = i+1; j < n; j++) {\n      A[i][j] = 0;\n    }\n    A[i][i] = 1;\n  }"}
{"kernel": "lu", "line": 33, "code": "for (j = 0; j <= i; j++)\n  A[i][j] = (DATA_TYPE)(-j % n) / n + 1;"}
{"kernel": "lu", "line": 35, "code": "for (j = i+1; j < n; j++) {\n  A[i][j] = 0;\n}"}
{"kernel": "lu", "line": 45, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "lu", "line": 46, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "lu", "line": 48, "code": "for (t = 0; t < n; ++t)\n  for (r = 0; r < n; ++r)\n    for (s = 0; s < n; ++s)\n      (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "lu", "line": 49, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "lu", "line": 50, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "lu", "line": 52, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "lu", "line": 53, "code": "for (s = 0; s < n; ++s)\n  A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "lu", "line": 71, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n  }"}
{"kernel": "lu", "line": 72, "code": "for (j = 0; j < n; j++) {\n  if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n}"}
{"kernel": "lu", "line": 90, "code": "for (i = 0; i < _PB_N; i++) {\n  for (j = 0; j <i; j++) {\n     for (k = 0; k < j; k++) {\n        A[i][j] -= A[i][k] * A[k][j];\n     }\n      A[i][j] /= A[j][j];\n  }\n for (j = i; j < _PB_N; j++) {\n     for (k = 0; k < i; k++) {\n        A[i][j] -= A[i][k] * A[k][j];\n     }\n  }\n}"}
{"kernel": "lu", "line": 91, "code": "for (j = 0; j <i; j++) {\n   for (k = 0; k < j; k++) {\n      A[i][j] -= A[i][k] * A[k][j];\n   }\n    A[i][j] /= A[j][j];\n}"}
{"kernel": "lu", "line": 92, "code": "for (k = 0; k < j; k++) {\n   A[i][j] -= A[i][k] * A[k][j];\n}"}
{"kernel": "lu", "line": 97, "code": "for (j = i; j < _PB_N; j++) {\n    for (k = 0; k < i; k++) {\n       A[i][j] -= A[i][k] * A[k][j];\n    }\n }"}
{"kernel": "lu", "line": 98, "code": "for (k = 0; k < i; k++) {\n   A[i][j] -= A[i][k] * A[k][j];\n}"}
{"kernel": "ludcmp", "line": 35, "code": "for (i = 0; i < n; i++)\n  {\n    x[i] = 0;\n    y[i] = 0;\n    b[i] = (i+1)/fn/2.0 + 4;\n  }"}
{"kernel": "ludcmp", "line": 42, "code": "for (i = 0; i < n; i++)\n  {\n    for (j = 0; j <= i; j++)\n      A[i][j] = (DATA_TYPE)(-j % n) / n + 1;\n    for (j = i+1; j < n; j++) {\n      A[i][j] = 0;\n    }\n    A[i][i] = 1;\n  }"}
{"kernel": "ludcmp", "line": 44, "code": "for (j = 0; j <= i; j++)\n  A[i][j] = (DATA_TYPE)(-j % n) / n + 1;"}
{"kernel": "ludcmp", "line": 46, "code": "for (j = i+1; j < n; j++) {\n  A[i][j] = 0;\n}"}
{"kernel": "ludcmp", "line": 56, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "ludcmp", "line": 57, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] = 0;"}
{"kernel": "ludcmp", "line": 59, "code": "for (t = 0; t < n; ++t)\n  for (r = 0; r < n; ++r)\n    for (s = 0; s < n; ++s)\n      (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "ludcmp", "line": 60, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "ludcmp", "line": 61, "code": "for (s = 0; s < n; ++s)\n  (POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];"}
{"kernel": "ludcmp", "line": 63, "code": "for (r = 0; r < n; ++r)\n  for (s = 0; s < n; ++s)\n    A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "ludcmp", "line": 64, "code": "for (s = 0; s < n; ++s)\n  A[r][s] = (POLYBENCH_ARRAY(B))[r][s];"}
{"kernel": "ludcmp", "line": 82, "code": "for (i = 0; i < n; i++) {\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\n}"}
{"kernel": "ludcmp", "line": 105, "code": "for (i = 0; i < _PB_N; i++) {\n  for (j = 0; j <i; j++) {\n     w = A[i][j];\n     for (k = 0; k < j; k++) {\n        w -= A[i][k] * A[k][j];\n     }\n      A[i][j] = w / A[j][j];\n  }\n for (j = i; j < _PB_N; j++) {\n     w = A[i][j];\n     for (k = 0; k < i; k++) {\n        w -= A[i][k] * A[k][j];\n     }\n     A[i][j] = w;\n  }\n}"}
{"kernel": "ludcmp", "line": 106, "code": "for (j = 0; j <i; j++) {\n   w = A[i][j];\n   for (k = 0; k < j; k++) {\n      w -= A[i][k] * A[k][j];\n   }\n    A[i][j] = w / A[j][j];\n}"}
{"kernel": "ludcmp", "line": 108, "code": "for (k = 0; k < j; k++) {\n   w -= A[i][k] * A[k][j];\n}"}
{"kernel": "ludcmp", "line": 113, "code": "for (j = i; j < _PB_N; j++) {\n    w = A[i][j];\n    for (k = 0; k < i; k++) {\n       w -= A[i][k] * A[k][j];\n    }\n    A[i][j] = w;\n }"}
{"kernel": "ludcmp", "line": 115, "code": "for (k = 0; k < i; k++) {\n   w -= A[i][k] * A[k][j];\n}"}
{"kernel": "ludcmp", "line": 122, "code": "for (i = 0; i < _PB_N; i++) {\n   w = b[i];\n   for (j = 0; j < i; j++)\n      w -= A[i][j] * y[j];\n   y[i] = w;\n}"}
{"kernel": "ludcmp", "line": 124, "code": "for (j = 0; j < i; j++)\n   w -= A[i][j] * y[j];"}
{"kernel": "ludcmp", "line": 129, "code": " for (i = _PB_N-1; i >=0; i--) {\n   w = y[i];\n   for (j = i+1; j < _PB_N; j++)\n      w -= A[i][j] * x[j];\n   x[i] = w / A[i][i];\n}"}
{"kernel": "ludcmp", "line": 131, "code": "for (j = i+1; j < _PB_N; j++)\n   w -= A[i][j] * x[j];"}
{"kernel": "trisolv", "line": 33, "code": "for (i = 0; i < n; i++)\n  {\n    x[i] = - 999;\n    b[i] =  i ;\n    for (j = 0; j <= i; j++)\n      L[i][j] = (DATA_TYPE) (i+n-j+1)*2/n;\n  }"}
{"kernel": "trisolv", "line": 37, "code": "for (j = 0; j <= i; j++)\n  L[i][j] = (DATA_TYPE) (i+n-j+1)*2/n;"}
{"kernel": "trisolv", "line": 54, "code": "for (i = 0; i < n; i++) {\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\n  if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n}"}
{"kernel": "trisolv", "line": 74, "code": "for (i = 0; i < _PB_N; i++)\n  {\n    x[i] = b[i];\n    for (j = 0; j <i; j++)\n      x[i] -= L[i][j] * x[j];\n    x[i] = x[i] / L[i][i];\n  }"}
{"kernel": "trisolv", "line": 77, "code": "for (j = 0; j <i; j++)\n  x[i] -= L[i][j] * x[j];"}
{"kernel": "deriche", "line": 35, "code": "for (i = 0; i < w; i++)\n   for (j = 0; j < h; j++)\n      imgIn[i][j] = (DATA_TYPE) ((313*i+991*j)%65536) / 65535.0f;"}
{"kernel": "deriche", "line": 36, "code": "for (j = 0; j < h; j++)\n   imgIn[i][j] = (DATA_TYPE) ((313*i+991*j)%65536) / 65535.0f;"}
{"kernel": "deriche", "line": 52, "code": "for (i = 0; i < w; i++)\n  for (j = 0; j < h; j++) {\n    if ((i * h + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, imgOut[i][j]);\n  }"}
{"kernel": "deriche", "line": 53, "code": "for (j = 0; j < h; j++) {\n  if ((i * h + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, imgOut[i][j]);\n}"}
{"kernel": "deriche", "line": 92, "code": "for (i=0; i<_PB_W; i++) {\n     ym1 = SCALAR_VAL(0.0);\n     ym2 = SCALAR_VAL(0.0);\n     xm1 = SCALAR_VAL(0.0);\n     for (j=0; j<_PB_H; j++) {\n         y1[i][j] = a1*imgIn[i][j] + a2*xm1 + b1*ym1 + b2*ym2;\n         xm1 = imgIn[i][j];\n         ym2 = ym1;\n         ym1 = y1[i][j];\n     }\n }"}
{"kernel": "deriche", "line": 96, "code": "for (j=0; j<_PB_H; j++) {\n    y1[i][j] = a1*imgIn[i][j] + a2*xm1 + b1*ym1 + b2*ym2;\n    xm1 = imgIn[i][j];\n    ym2 = ym1;\n    ym1 = y1[i][j];\n}"}
{"kernel": "deriche", "line": 104, "code": "for (i=0; i<_PB_W; i++) {\n    yp1 = SCALAR_VAL(0.0);\n    yp2 = SCALAR_VAL(0.0);\n    xp1 = SCALAR_VAL(0.0);\n    xp2 = SCALAR_VAL(0.0);\n    for (j=_PB_H-1; j>=0; j--) {\n        y2[i][j] = a3*xp1 + a4*xp2 + b1*yp1 + b2*yp2;\n        xp2 = xp1;\n        xp1 = imgIn[i][j];\n        yp2 = yp1;\n        yp1 = y2[i][j];\n    }\n}"}
{"kernel": "deriche", "line": 109, "code": "for (j=_PB_H-1; j>=0; j--) {\n    y2[i][j] = a3*xp1 + a4*xp2 + b1*yp1 + b2*yp2;\n    xp2 = xp1;\n    xp1 = imgIn[i][j];\n    yp2 = yp1;\n    yp1 = y2[i][j];\n}"}
{"kernel": "deriche", "line": 118, "code": "for (i=0; i<_PB_W; i++)\n    for (j=0; j<_PB_H; j++) {\n        imgOut[i][j] = c1 * (y1[i][j] + y2[i][j]);\n    }"}
{"kernel": "deriche", "line": 119, "code": "for (j=0; j<_PB_H; j++) {\n    imgOut[i][j] = c1 * (y1[i][j] + y2[i][j]);\n}"}
{"kernel": "deriche", "line": 123, "code": "for (j=0; j<_PB_H; j++) {\n    tm1 = SCALAR_VAL(0.0);\n    ym1 = SCALAR_VAL(0.0);\n    ym2 = SCALAR_VAL(0.0);\n    for (i=0; i<_PB_W; i++) {\n        y1[i][j] = a5*imgOut[i][j] + a6*tm1 + b1*ym1 + b2*ym2;\n        tm1 = imgOut[i][j];\n        ym2 = ym1;\n        ym1 = y1 [i][j];\n    }\n}"}
{"kernel": "deriche", "line": 127, "code": "for (i=0; i<_PB_W; i++) {\n    y1[i][j] = a5*imgOut[i][j] + a6*tm1 + b1*ym1 + b2*ym2;\n    tm1 = imgOut[i][j];\n    ym2 = ym1;\n    ym1 = y1 [i][j];\n}"}
{"kernel": "deriche", "line": 136, "code": "for (j=0; j<_PB_H; j++) {\n    tp1 = SCALAR_VAL(0.0);\n    tp2 = SCALAR_VAL(0.0);\n    yp1 = SCALAR_VAL(0.0);\n    yp2 = SCALAR_VAL(0.0);\n    for (i=_PB_W-1; i>=0; i--) {\n        y2[i][j] = a7*tp1 + a8*tp2 + b1*yp1 + b2*yp2;\n        tp2 = tp1;\n        tp1 = imgOut[i][j];\n        yp2 = yp1;\n        yp1 = y2[i][j];\n    }\n}"}
{"kernel": "deriche", "line": 141, "code": "for (i=_PB_W-1; i>=0; i--) {\n    y2[i][j] = a7*tp1 + a8*tp2 + b1*yp1 + b2*yp2;\n    tp2 = tp1;\n    tp1 = imgOut[i][j];\n    yp2 = yp1;\n    yp1 = y2[i][j];\n}"}
{"kernel": "deriche", "line": 150, "code": "for (i=0; i<_PB_W; i++)\n    for (j=0; j<_PB_H; j++)\n        imgOut[i][j] = c2*(y1[i][j] + y2[i][j]);"}
{"kernel": "deriche", "line": 151, "code": "for (j=0; j<_PB_H; j++)\n    imgOut[i][j] = c2*(y1[i][j] + y2[i][j]);"}
{"kernel": "floyd-warshall", "line": 31, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    path[i][j] = i*j%7+1;\n    if ((i+j)%13 == 0 || (i+j)%7==0 || (i+j)%11 == 0)\n       path[i][j] = 999;\n  }"}
{"kernel": "floyd-warshall", "line": 32, "code": "for (j = 0; j < n; j++) {\n  path[i][j] = i*j%7+1;\n  if ((i+j)%13 == 0 || (i+j)%7==0 || (i+j)%11 == 0)\n     path[i][j] = 999;\n}"}
{"kernel": "floyd-warshall", "line": 51, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, path[i][j]);\n  }"}
{"kernel": "floyd-warshall", "line": 52, "code": "for (j = 0; j < n; j++) {\n  if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, path[i][j]);\n}"}
{"kernel": "floyd-warshall", "line": 70, "code": "for (k = 0; k < _PB_N; k++)\n  {\n    for(i = 0; i < _PB_N; i++)\n      for (j = 0; j < _PB_N; j++)\n        path[i][j] = path[i][j] < path[i][k] + path[k][j] ?\n          path[i][j] : path[i][k] + path[k][j];\n  }"}
{"kernel": "floyd-warshall", "line": 72, "code": "for(i = 0; i < _PB_N; i++)\n  for (j = 0; j < _PB_N; j++)\n    path[i][j] = path[i][j] < path[i][k] + path[k][j] ?\n      path[i][j] : path[i][k] + path[k][j];"}
{"kernel": "floyd-warshall", "line": 73, "code": "for (j = 0; j < _PB_N; j++)\n  path[i][j] = path[i][j] < path[i][k] + path[k][j] ?\n    path[i][j] : path[i][k] + path[k][j];"}
{"kernel": "nussinov", "line": 38, "code": "for (i=0; i <n; i++) {\n   seq[i] = (base)((i+1)%4);\n}"}
{"kernel": "nussinov", "line": 42, "code": "for (i=0; i <n; i++)\n   for (j=0; j <n; j++)\n     table[i][j] = 0;"}
{"kernel": "nussinov", "line": 43, "code": "for (j=0; j <n; j++)\n  table[i][j] = 0;"}
{"kernel": "nussinov", "line": 60, "code": "for (i = 0; i < n; i++) {\n  for (j = i; j < n; j++) {\n    if (t % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, table[i][j]);\n    t++;\n  }\n}"}
{"kernel": "nussinov", "line": 61, "code": "for (j = i; j < n; j++) {\n  if (t % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, table[i][j]);\n  t++;\n}"}
{"kernel": "nussinov", "line": 86, "code": "for (i = _PB_N-1; i >= 0; i--) {\n for (j=i+1; j<_PB_N; j++) {\n\n  if (j-1>=0)\n     table[i][j] = max_score(table[i][j], table[i][j-1]);\n  if (i+1<_PB_N)\n     table[i][j] = max_score(table[i][j], table[i+1][j]);\n\n  if (j-1>=0 && i+1<_PB_N) {\n    /* don't allow adjacent elements to bond */\n    if (i<j-1)\n       table[i][j] = max_score(table[i][j], table[i+1][j-1]+match(seq[i], seq[j]));\n    else\n       table[i][j] = max_score(table[i][j], table[i+1][j-1]);\n  }\n\n  for (k=i+1; k<j; k++) {\n     table[i][j] = max_score(table[i][j], table[i][k] + table[k+1][j]);\n  }\n }\n}"}
{"kernel": "nussinov", "line": 87, "code": "for (j=i+1; j<_PB_N; j++) {\n\n if (j-1>=0)\n    table[i][j] = max_score(table[i][j], table[i][j-1]);\n if (i+1<_PB_N)\n    table[i][j] = max_score(table[i][j], table[i+1][j]);\n\n if (j-1>=0 && i+1<_PB_N) {\n   /* don't allow adjacent elements to bond */\n   if (i<j-1)\n      table[i][j] = max_score(table[i][j], table[i+1][j-1]+match(seq[i], seq[j]));\n   else\n      table[i][j] = max_score(table[i][j], table[i+1][j-1]);\n }\n\n for (k=i+1; k<j; k++) {\n    table[i][j] = max_score(table[i][j], table[i][k] + table[k+1][j]);\n }\n}"}
{"kernel": "nussinov", "line": 102, "code": "for (k=i+1; k<j; k++) {\n   table[i][j] = max_score(table[i][j], table[i][k] + table[k+1][j]);\n}"}
{"kernel": "adi", "line": 31, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    {\n      u[i][j] =  (DATA_TYPE)(i + n-j) / n;\n    }"}
{"kernel": "adi", "line": 32, "code": "for (j = 0; j < n; j++)\n  {\n    u[i][j] =  (DATA_TYPE)(i + n-j) / n;\n  }"}
{"kernel": "adi", "line": 50, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, u[i][j]);\n  }"}
{"kernel": "adi", "line": 51, "code": "for (j = 0; j < n; j++) {\n  if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, u[i][j]);\n}"}
{"kernel": "adi", "line": 96, "code": "for (t=1; t<=_PB_TSTEPS; t++) {\n   //Column Sweep\n   for (i=1; i<_PB_N-1; i++) {\n     v[0][i] = SCALAR_VAL(1.0);\n     p[i][0] = SCALAR_VAL(0.0);\n     q[i][0] = v[0][i];\n     for (j=1; j<_PB_N-1; j++) {\n       p[i][j] = -c / (a*p[i][j-1]+b);\n       q[i][j] = (-d*u[j][i-1]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*d)*u[j][i] - f*u[j][i+1]-a*q[i][j-1])/(a*p[i][j-1]+b);\n     }\n\n     v[_PB_N-1][i] = SCALAR_VAL(1.0);\n     for (j=_PB_N-2; j>=1; j--) {\n       v[j][i] = p[i][j] * v[j+1][i] + q[i][j];\n     }\n   }\n   //Row Sweep\n   for (i=1; i<_PB_N-1; i++) {\n     u[i][0] = SCALAR_VAL(1.0);\n     p[i][0] = SCALAR_VAL(0.0);\n     q[i][0] = u[i][0];\n     for (j=1; j<_PB_N-1; j++) {\n       p[i][j] = -f / (d*p[i][j-1]+e);\n       q[i][j] = (-a*v[i-1][j]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*a)*v[i][j] - c*v[i+1][j]-d*q[i][j-1])/(d*p[i][j-1]+e);\n     }\n     u[i][_PB_N-1] = SCALAR_VAL(1.0);\n     for (j=_PB_N-2; j>=1; j--) {\n       u[i][j] = p[i][j] * u[i][j+1] + q[i][j];\n     }\n   }\n }"}
{"kernel": "adi", "line": 98, "code": "for (i=1; i<_PB_N-1; i++) {\n  v[0][i] = SCALAR_VAL(1.0);\n  p[i][0] = SCALAR_VAL(0.0);\n  q[i][0] = v[0][i];\n  for (j=1; j<_PB_N-1; j++) {\n    p[i][j] = -c / (a*p[i][j-1]+b);\n    q[i][j] = (-d*u[j][i-1]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*d)*u[j][i] - f*u[j][i+1]-a*q[i][j-1])/(a*p[i][j-1]+b);\n  }\n\n  v[_PB_N-1][i] = SCALAR_VAL(1.0);\n  for (j=_PB_N-2; j>=1; j--) {\n    v[j][i] = p[i][j] * v[j+1][i] + q[i][j];\n  }\n}"}
{"kernel": "adi", "line": 102, "code": "for (j=1; j<_PB_N-1; j++) {\n  p[i][j] = -c / (a*p[i][j-1]+b);\n  q[i][j] = (-d*u[j][i-1]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*d)*u[j][i] - f*u[j][i+1]-a*q[i][j-1])/(a*p[i][j-1]+b);\n}"}
{"kernel": "adi", "line": 108, "code": "for (j=_PB_N-2; j>=1; j--) {\n  v[j][i] = p[i][j] * v[j+1][i] + q[i][j];\n}"}
{"kernel": "adi", "line": 113, "code": "for (i=1; i<_PB_N-1; i++) {\n  u[i][0] = SCALAR_VAL(1.0);\n  p[i][0] = SCALAR_VAL(0.0);\n  q[i][0] = u[i][0];\n  for (j=1; j<_PB_N-1; j++) {\n    p[i][j] = -f / (d*p[i][j-1]+e);\n    q[i][j] = (-a*v[i-1][j]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*a)*v[i][j] - c*v[i+1][j]-d*q[i][j-1])/(d*p[i][j-1]+e);\n  }\n  u[i][_PB_N-1] = SCALAR_VAL(1.0);\n  for (j=_PB_N-2; j>=1; j--) {\n    u[i][j] = p[i][j] * u[i][j+1] + q[i][j];\n  }\n}"}
{"kernel": "adi", "line": 117, "code": "for (j=1; j<_PB_N-1; j++) {\n  p[i][j] = -f / (d*p[i][j-1]+e);\n  q[i][j] = (-a*v[i-1][j]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*a)*v[i][j] - c*v[i+1][j]-d*q[i][j-1])/(d*p[i][j-1]+e);\n}"}
{"kernel": "adi", "line": 122, "code": "for (j=_PB_N-2; j>=1; j--) {\n  u[i][j] = p[i][j] * u[i][j+1] + q[i][j];\n}"}
{"kernel": "fdtd-2d", "line": 36, "code": "for (i = 0; i < tmax; i++)\n  _fict_[i] = (DATA_TYPE) i;"}
{"kernel": "fdtd-2d", "line": 38, "code": "for (i = 0; i < nx; i++)\n  for (j = 0; j < ny; j++)\n    {\n      ex[i][j] = ((DATA_TYPE) i*(j+1)) / nx;\n      ey[i][j] = ((DATA_TYPE) i*(j+2)) / ny;\n      hz[i][j] = ((DATA_TYPE) i*(j+3)) / nx;\n    }"}
{"kernel": "fdtd-2d", "line": 39, "code": "for (j = 0; j < ny; j++)\n  {\n    ex[i][j] = ((DATA_TYPE) i*(j+1)) / nx;\n    ey[i][j] = ((DATA_TYPE) i*(j+2)) / ny;\n    hz[i][j] = ((DATA_TYPE) i*(j+3)) / nx;\n  }"}
{"kernel": "fdtd-2d", "line": 61, "code": "for (i = 0; i < nx; i++)\n  for (j = 0; j < ny; j++) {\n    if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ex[i][j]);\n  }"}
{"kernel": "fdtd-2d", "line": 62, "code": "for (j = 0; j < ny; j++) {\n  if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ex[i][j]);\n}"}
{"kernel": "fdtd-2d", "line": 70, "code": "for (i = 0; i < nx; i++)\n  for (j = 0; j < ny; j++) {\n    if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ey[i][j]);\n  }"}
{"kernel": "fdtd-2d", "line": 71, "code": "for (j = 0; j < ny; j++) {\n  if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ey[i][j]);\n}"}
{"kernel": "fdtd-2d", "line": 78, "code": "for (i = 0; i < nx; i++)\n  for (j = 0; j < ny; j++) {\n    if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, hz[i][j]);\n  }"}
{"kernel": "fdtd-2d", "line": 79, "code": "for (j = 0; j < ny; j++) {\n  if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, hz[i][j]);\n}"}
{"kernel": "fdtd-2d", "line": 102, "code": "for(t = 0; t < _PB_TMAX; t++)\n  {\n    for (j = 0; j < _PB_NY; j++)\n      ey[0][j] = _fict_[t];\n    for (i = 1; i < _PB_NX; i++)\n      for (j = 0; j < _PB_NY; j++)\n        ey[i][j] = ey[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i-1][j]);\n    for (i = 0; i < _PB_NX; i++)\n      for (j = 1; j < _PB_NY; j++)\n        ex[i][j] = ex[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i][j-1]);\n    for (i = 0; i < _PB_NX - 1; i++)\n      for (j = 0; j < _PB_NY - 1; j++)\n        hz[i][j] = hz[i][j] - SCALAR_VAL(0.7)*  (ex[i][j+1] - ex[i][j] +\n                                     ey[i+1][j] - ey[i][j]);\n  }"}
{"kernel": "fdtd-2d", "line": 104, "code": "for (j = 0; j < _PB_NY; j++)\n  ey[0][j] = _fict_[t];"}
{"kernel": "fdtd-2d", "line": 106, "code": "for (i = 1; i < _PB_NX; i++)\n  for (j = 0; j < _PB_NY; j++)\n    ey[i][j] = ey[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i-1][j]);"}
{"kernel": "fdtd-2d", "line": 107, "code": "for (j = 0; j < _PB_NY; j++)\n  ey[i][j] = ey[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i-1][j]);"}
{"kernel": "fdtd-2d", "line": 109, "code": "for (i = 0; i < _PB_NX; i++)\n  for (j = 1; j < _PB_NY; j++)\n    ex[i][j] = ex[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i][j-1]);"}
{"kernel": "fdtd-2d", "line": 110, "code": "for (j = 1; j < _PB_NY; j++)\n  ex[i][j] = ex[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i][j-1]);"}
{"kernel": "fdtd-2d", "line": 112, "code": "for (i = 0; i < _PB_NX - 1; i++)\n  for (j = 0; j < _PB_NY - 1; j++)\n    hz[i][j] = hz[i][j] - SCALAR_VAL(0.7)*  (ex[i][j+1] - ex[i][j] +\n                                 ey[i+1][j] - ey[i][j]);"}
{"kernel": "fdtd-2d", "line": 113, "code": "for (j = 0; j < _PB_NY - 1; j++)\n  hz[i][j] = hz[i][j] - SCALAR_VAL(0.7)*  (ex[i][j+1] - ex[i][j] +\n                               ey[i+1][j] - ey[i][j]);"}
{"kernel": "heat-3d", "line": 32, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    for (k = 0; k < n; k++)\n      A[i][j][k] = B[i][j][k] = (DATA_TYPE) (i + j + (n-k))* 10 / (n);"}
{"kernel": "heat-3d", "line": 33, "code": "for (j = 0; j < n; j++)\n  for (k = 0; k < n; k++)\n    A[i][j][k] = B[i][j][k] = (DATA_TYPE) (i + j + (n-k))* 10 / (n);"}
{"kernel": "heat-3d", "line": 34, "code": "for (k = 0; k < n; k++)\n  A[i][j][k] = B[i][j][k] = (DATA_TYPE) (i + j + (n-k))* 10 / (n);"}
{"kernel": "heat-3d", "line": 50, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    for (k = 0; k < n; k++) {\n       if ((i * n * n + j * n + k) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n       fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n    }"}
{"kernel": "heat-3d", "line": 51, "code": "for (j = 0; j < n; j++)\n  for (k = 0; k < n; k++) {\n     if ((i * n * n + j * n + k) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n     fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n  }"}
{"kernel": "heat-3d", "line": 52, "code": "for (k = 0; k < n; k++) {\n   if ((i * n * n + j * n + k) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n   fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\n}"}
{"kernel": "heat-3d", "line": 72, "code": "for (t = 1; t <= TSTEPS; t++) {\n    for (i = 1; i < _PB_N-1; i++) {\n        for (j = 1; j < _PB_N-1; j++) {\n            for (k = 1; k < _PB_N-1; k++) {\n                B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\n                             + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\n                             + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\n                             + A[i][j][k];\n            }\n        }\n    }\n    for (i = 1; i < _PB_N-1; i++) {\n       for (j = 1; j < _PB_N-1; j++) {\n           for (k = 1; k < _PB_N-1; k++) {\n               A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\n                            + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\n                            + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\n                            + B[i][j][k];\n           }\n       }\n   }\n}"}
{"kernel": "heat-3d", "line": 73, "code": "for (i = 1; i < _PB_N-1; i++) {\n    for (j = 1; j < _PB_N-1; j++) {\n        for (k = 1; k < _PB_N-1; k++) {\n            B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\n                         + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\n                         + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\n                         + A[i][j][k];\n        }\n    }\n}"}
{"kernel": "heat-3d", "line": 74, "code": "for (j = 1; j < _PB_N-1; j++) {\n    for (k = 1; k < _PB_N-1; k++) {\n        B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\n                     + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\n                     + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\n                     + A[i][j][k];\n    }\n}"}
{"kernel": "heat-3d", "line": 75, "code": "for (k = 1; k < _PB_N-1; k++) {\n    B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\n                 + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\n                 + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\n                 + A[i][j][k];\n}"}
{"kernel": "heat-3d", "line": 83, "code": " for (i = 1; i < _PB_N-1; i++) {\n    for (j = 1; j < _PB_N-1; j++) {\n        for (k = 1; k < _PB_N-1; k++) {\n            A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\n                         + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\n                         + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\n                         + B[i][j][k];\n        }\n    }\n}"}
{"kernel": "heat-3d", "line": 84, "code": "for (j = 1; j < _PB_N-1; j++) {\n    for (k = 1; k < _PB_N-1; k++) {\n        A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\n                     + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\n                     + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\n                     + B[i][j][k];\n    }\n}"}
{"kernel": "heat-3d", "line": 85, "code": "for (k = 1; k < _PB_N-1; k++) {\n    A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\n                 + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\n                 + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\n                 + B[i][j][k];\n}"}
{"kernel": "jacobi-1d", "line": 32, "code": "for (i = 0; i < n; i++)\n    {\n      A[i] = ((DATA_TYPE) i+ 2) / n;\n      B[i] = ((DATA_TYPE) i+ 3) / n;\n    }"}
{"kernel": "jacobi-1d", "line": 51, "code": "for (i = 0; i < n; i++)\n  {\n    if (i % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i]);\n  }"}
{"kernel": "jacobi-1d", "line": 72, "code": "for (t = 0; t < _PB_TSTEPS; t++)\n  {\n    for (i = 1; i < _PB_N - 1; i++)\n      B[i] = 0.33333 * (A[i-1] + A[i] + A[i + 1]);\n    for (i = 1; i < _PB_N - 1; i++)\n      A[i] = 0.33333 * (B[i-1] + B[i] + B[i + 1]);\n  }"}
{"kernel": "jacobi-1d", "line": 74, "code": "for (i = 1; i < _PB_N - 1; i++)\n  B[i] = 0.33333 * (A[i-1] + A[i] + A[i + 1]);"}
{"kernel": "jacobi-1d", "line": 76, "code": "for (i = 1; i < _PB_N - 1; i++)\n  A[i] = 0.33333 * (B[i-1] + B[i] + B[i + 1]);"}
{"kernel": "jacobi-2d", "line": 32, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    {\n      A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\n      B[i][j] = ((DATA_TYPE) i*(j+3) + 3) / n;\n    }"}
{"kernel": "jacobi-2d", "line": 33, "code": "for (j = 0; j < n; j++)\n  {\n    A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\n    B[i][j] = ((DATA_TYPE) i*(j+3) + 3) / n;\n  }"}
{"kernel": "jacobi-2d", "line": 52, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n  }"}
{"kernel": "jacobi-2d", "line": 53, "code": "for (j = 0; j < n; j++) {\n  if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n}"}
{"kernel": "jacobi-2d", "line": 73, "code": "for (t = 0; t < _PB_TSTEPS; t++)\n  {\n    for (i = 1; i < _PB_N - 1; i++)\n      for (j = 1; j < _PB_N - 1; j++)\n        B[i][j] = SCALAR_VAL(0.2) * (A[i][j] + A[i][j-1] + A[i][1+j] + A[1+i][j] + A[i-1][j]);\n    for (i = 1; i < _PB_N - 1; i++)\n      for (j = 1; j < _PB_N - 1; j++)\n        A[i][j] = SCALAR_VAL(0.2) * (B[i][j] + B[i][j-1] + B[i][1+j] + B[1+i][j] + B[i-1][j]);\n  }"}
{"kernel": "jacobi-2d", "line": 75, "code": "for (i = 1; i < _PB_N - 1; i++)\n  for (j = 1; j < _PB_N - 1; j++)\n    B[i][j] = SCALAR_VAL(0.2) * (A[i][j] + A[i][j-1] + A[i][1+j] + A[1+i][j] + A[i-1][j]);"}
{"kernel": "jacobi-2d", "line": 76, "code": "for (j = 1; j < _PB_N - 1; j++)\n  B[i][j] = SCALAR_VAL(0.2) * (A[i][j] + A[i][j-1] + A[i][1+j] + A[1+i][j] + A[i-1][j]);"}
{"kernel": "jacobi-2d", "line": 78, "code": "for (i = 1; i < _PB_N - 1; i++)\n  for (j = 1; j < _PB_N - 1; j++)\n    A[i][j] = SCALAR_VAL(0.2) * (B[i][j] + B[i][j-1] + B[i][1+j] + B[1+i][j] + B[i-1][j]);"}
{"kernel": "jacobi-2d", "line": 79, "code": "for (j = 1; j < _PB_N - 1; j++)\n  A[i][j] = SCALAR_VAL(0.2) * (B[i][j] + B[i][j-1] + B[i][1+j] + B[1+i][j] + B[i-1][j]);"}
{"kernel": "seidel-2d", "line": 31, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++)\n    A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;"}
{"kernel": "seidel-2d", "line": 32, "code": "for (j = 0; j < n; j++)\n  A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;"}
{"kernel": "seidel-2d", "line": 48, "code": "for (i = 0; i < n; i++)\n  for (j = 0; j < n; j++) {\n    if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n    fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n  }"}
{"kernel": "seidel-2d", "line": 49, "code": "for (j = 0; j < n; j++) {\n  if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \"\\n\");\n  fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\n}"}
{"kernel": "seidel-2d", "line": 68, "code": "for (t = 0; t <= _PB_TSTEPS - 1; t++)\n  for (i = 1; i<= _PB_N - 2; i++)\n    for (j = 1; j <= _PB_N - 2; j++)\n      A[i][j] = (A[i-1][j-1] + A[i-1][j] + A[i-1][j+1]\n                 + A[i][j-1] + A[i][j] + A[i][j+1]\n                 + A[i+1][j-1] + A[i+1][j] + A[i+1][j+1])/SCALAR_VAL(9.0);"}
{"kernel": "seidel-2d", "line": 69, "code": "for (i = 1; i<= _PB_N - 2; i++)\n  for (j = 1; j <= _PB_N - 2; j++)\n    A[i][j] = (A[i-1][j-1] + A[i-1][j] + A[i-1][j+1]\n               + A[i][j-1] + A[i][j] + A[i][j+1]\n               + A[i+1][j-1] + A[i+1][j] + A[i+1][j+1])/SCALAR_VAL(9.0);"}
{"kernel": "seidel-2d", "line": 70, "code": "for (j = 1; j <= _PB_N - 2; j++)\n  A[i][j] = (A[i-1][j-1] + A[i-1][j] + A[i-1][j+1]\n             + A[i][j-1] + A[i][j] + A[i][j+1]\n             + A[i+1][j-1] + A[i+1][j] + A[i+1][j+1])/SCALAR_VAL(9.0);"}


In [ ]:
%%bash
set -e
cd /content/OMPar
PYTHONPATH=/content/OMPar /content/venv/bin/python /content/ompar_run.py \
  --model_weights /content/weights /content/loops.jsonl /content/results.jsonl
wc -l /content/loops.jsonl /content/results.jsonl


In [ ]:
from google.colab import files
files.download('/content/results.jsonl')
